# DID Analysis: How ChatGPT Affected Crowdfunding Success

## Research Question
Did the launch of ChatGPT (Nov 30, 2022) change how text quality affects crowdfunding success?

## Method
- **Treatment**: ChatGPT Launch (November 30, 2022)
- **Outcome**: Log pledged amount (USD)
- **Sample**: Projects with >600 words
- **Comparison**: Kickstarter vs Indiegogo platforms


In [2]:
# Step 1: Load libraries and data
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import re
import warnings
warnings.filterwarnings('ignore')

# Load data with AI detection scores
df = pd.read_pickle('final_with_deberta_ai_score_20251003_151656.pkl')
print(f"Loaded {len(df):,} projects")

Loaded 64,488 projects


In [3]:
import parquet

df.to_parquet('intermediate_with_text_quality_and_ai.parquet', engine='fastparquet', index=False)

In [4]:
# Step 2: Create key variables

# Fix dates
def clean_timezone(date_str):
    if pd.isna(date_str):
        return pd.NaT
    return re.sub(r'[+-]\d{2}:\d{2}$', '', str(date_str))

df['created_at_parsed'] = pd.to_datetime(df['created_at'].apply(clean_timezone), errors='coerce')

# Create outcome variable: log pledged amount
ks_mask = df['platform'] == 'Kickstarter'
ig_mask = df['platform'] == 'Indiegogo'

df['pledged_amount_usd'] = 0
df.loc[ks_mask, 'pledged_amount_usd'] = (df.loc[ks_mask, 'percent_funded'] / 100 * df.loc[ks_mask, 'goal']).fillna(0)
df.loc[ig_mask, 'pledged_amount_usd'] = (df.loc[ig_mask, 'funds_raised_percent'] / 100 * df.loc[ig_mask, 'goal']).fillna(0)
df['log_pledged_amount'] = np.log(df['pledged_amount_usd'].clip(lower=1))

print("✅ Created outcome variable: log_pledged_amount")


✅ Created outcome variable: log_pledged_amount


In [5]:
# Step 3: Create unified categories

def create_unified_categories(df):
    # Using your exact category specification
    
    # Kickstarter mapping
    ks_mapping = {
        # 1. Creative / Visual Arts
        'Illustration': 'Creative_Visual_Arts', 'Digital Art': 'Creative_Visual_Arts', 'Art': 'Creative_Visual_Arts',
        'Animation': 'Creative_Visual_Arts', 'Painting': 'Creative_Visual_Arts', 'Sculpture': 'Creative_Visual_Arts',
        'Photography': 'Creative_Visual_Arts', 'Mixed Media': 'Creative_Visual_Arts', 'Design': 'Creative_Visual_Arts',
        'Fine Art': 'Creative_Visual_Arts', 'Public Art': 'Creative_Visual_Arts',
        
        # 2. Publishing & Writing
        'Comic Books': 'Publishing_Writing', 'Graphic Novels': 'Publishing_Writing', 'Fiction': 'Publishing_Writing',
        'Nonfiction': 'Publishing_Writing', 'Anthologies': 'Publishing_Writing', 'Zines': 'Publishing_Writing',
        'Poetry': 'Publishing_Writing', 'Literature': 'Publishing_Writing', 'Academic': 'Publishing_Writing',
        'Journals': 'Publishing_Writing', 'Comics': 'Publishing_Writing',
        
        # 3. Technology / Hardware / Gadgets
        'Product Design': 'Technology_Hardware', 'Hardware': 'Technology_Hardware', 'Apps': 'Technology_Hardware',
        'Gadgets': 'Technology_Hardware', 'DIY Electronics': 'Technology_Hardware', 'Wearables': 'Technology_Hardware',
        'Gaming Hardware': 'Technology_Hardware', 'Robotics': 'Technology_Hardware', '3D Printing': 'Technology_Hardware',
        'Software': 'Technology_Hardware', 'Technology': 'Technology_Hardware',
        
        # 4. Games & Toys
        'Playing Cards': 'Games_Toys', 'Tabletop Games': 'Games_Toys', 'Video Games': 'Games_Toys',
        'Mobile Games': 'Games_Toys', 'Live Games': 'Games_Toys', 'Puzzles': 'Games_Toys',
        'Toys': 'Games_Toys', 'Games': 'Games_Toys',
        
        # 5. Film, Music & Performance
        'Shorts': 'Film_Music_Performance', 'Drama': 'Film_Music_Performance', 'Comedy': 'Film_Music_Performance',
        'Horror': 'Film_Music_Performance', 'Documentary': 'Film_Music_Performance', 'Music': 'Film_Music_Performance',
        'Rock': 'Film_Music_Performance', 'Hip-Hop': 'Film_Music_Performance', 'Pop': 'Film_Music_Performance',
        'Jazz': 'Film_Music_Performance', 'Classical Music': 'Film_Music_Performance', 'Country & Folk': 'Film_Music_Performance',
        'Electronic Music': 'Film_Music_Performance', 'Indie Rock': 'Film_Music_Performance', 'Theater': 'Film_Music_Performance',
        'Performance Art': 'Film_Music_Performance', 'Festivals': 'Film_Music_Performance', 'Film & Video': 'Film_Music_Performance',
        'Musical': 'Film_Music_Performance', 'Webseries': 'Film_Music_Performance', 'Television': 'Film_Music_Performance',
        'Radio & Podcasts': 'Film_Music_Performance', 'Audio': 'Film_Music_Performance', 'Narrative Film': 'Film_Music_Performance',
        'Dance': 'Film_Music_Performance', 'Sound': 'Film_Music_Performance', 'Blues': 'Film_Music_Performance',
        'World Music': 'Film_Music_Performance', 'Latin': 'Film_Music_Performance', 'Metal': 'Film_Music_Performance',
        'Punk': 'Film_Music_Performance', 'R&B': 'Film_Music_Performance',
        
        # 6. Food & Lifestyle
        'Restaurants': 'Food_Lifestyle', 'Drinks': 'Food_Lifestyle', 'Food Trucks': 'Food_Lifestyle',
        'Vegan': 'Food_Lifestyle', 'Cookbooks': 'Food_Lifestyle', 'Candles': 'Food_Lifestyle',
        'Fashion': 'Food_Lifestyle', 'Apparel': 'Food_Lifestyle', 'Accessories': 'Food_Lifestyle',
        'Jewelry': 'Food_Lifestyle', 'Footwear': 'Food_Lifestyle', 'Cosmetics': 'Food_Lifestyle',
        'Food': 'Food_Lifestyle',
        
        # 7. Cause / Community / Education
        'Community Gardens': 'Community_Education', 'Faith': 'Community_Education', 'Social Practice': 'Community_Education',
        'Events': 'Community_Education', 'Community': 'Community_Education'
    }
    
    # Indiegogo mapping
    ig_mapping = {
        # 1. Creative / Visual Arts
        'Art': 'Creative_Visual_Arts', 'Photography': 'Creative_Visual_Arts', 'Dance & Theater': 'Creative_Visual_Arts',
        'Web Series & TV Shows': 'Creative_Visual_Arts',
        
        # 2. Publishing & Writing
        'Comics': 'Publishing_Writing', 'Writing & Publishing': 'Publishing_Writing', 'Blogs/Podcasts/Vlogs': 'Publishing_Writing',
        
        # 3. Technology / Hardware / Gadgets
        'Productivity': 'Technology_Hardware', 'Phones & Accessories': 'Technology_Hardware', 'Energy & Green Tech': 'Technology_Hardware',
        'Smart Home': 'Technology_Hardware', 'Computers': 'Technology_Hardware', 'IoT': 'Technology_Hardware',
        'Security': 'Technology_Hardware', 'Drones': 'Technology_Hardware', 'VR/AR': 'Technology_Hardware',
        'Transportation': 'Technology_Hardware',
        
        # 4. Games & Toys
        'Tabletop Games': 'Games_Toys', 'Video Games': 'Games_Toys', 'Toys': 'Games_Toys',
        
        # 5. Film, Music & Performance
        'Film': 'Film_Music_Performance', 'Music': 'Film_Music_Performance', 'Audio': 'Film_Music_Performance',
        'Web Series & TV': 'Film_Music_Performance', 'Dance & Theater': 'Film_Music_Performance',
        
        # 6. Food & Lifestyle
        'Food & Beverages': 'Food_Lifestyle', 'Fashion & Wearables': 'Food_Lifestyle', 'Home': 'Food_Lifestyle',
        'Beauty': 'Food_Lifestyle', 'Health & Fitness': 'Food_Lifestyle', 'Wellness': 'Food_Lifestyle',
        
        # 7. Cause / Community / Education
        'Education': 'Community_Education', 'Human Rights': 'Community_Education', 'Local Businesses': 'Community_Education',
        'Environment': 'Community_Education', 'Social Innovations': 'Community_Education', 'Culture': 'Community_Education'
    }
    
    # Apply mappings
    df['category_unified'] = 'Other'
    ks_mask = df['platform'] == 'Kickstarter'
    ig_mask = df['platform'] == 'Indiegogo'
    
    df.loc[ks_mask, 'category_unified'] = df.loc[ks_mask, 'category_name'].map(ks_mapping).fillna('Other')
    df.loc[ig_mask, 'category_unified'] = df.loc[ig_mask, 'category_parent_name'].map(ig_mapping).fillna('Other')
    
    return df

df = create_unified_categories(df)
print("✅ Created unified categories")
print(df['category_unified'].value_counts())


✅ Created unified categories
category_unified
Film_Music_Performance    15206
Other                     11130
Publishing_Writing         9461
Technology_Hardware        7332
Food_Lifestyle             7036
Creative_Visual_Arts       6832
Games_Toys                 5788
Community_Education        1703
Name: count, dtype: int64


In [6]:
# Step 4.5: Add Month and Year Controls for Date Fixed Effects
# This controls for seasonal patterns and year-to-year trends

print("📅 Adding month and year controls...")

df_analysis = df.copy()

# Create month and year variables for date controls
df_analysis['month'] = df_analysis['created_at_parsed'].dt.month
df_analysis['year'] = df_analysis['created_at_parsed'].dt.year
df_analysis['month_year'] = df_analysis['created_at_parsed'].dt.to_period('M').astype(str)

# Show date control information
print(f"   Date range: {df_analysis['created_at_parsed'].min()} to {df_analysis['created_at_parsed'].max()}")
print(f"   Years covered: {sorted(df_analysis['year'].unique())}")
print(f"   Months covered: {sorted(df_analysis['month'].unique())}")
print(f"   Unique month-year periods: {df_analysis['month_year'].nunique()}")

# Show distribution by year
print(f"\n📊 Projects by year:")
year_breakdown = df_analysis['year'].value_counts().sort_index()
print(year_breakdown)

# Show distribution by month
print(f"\n📊 Projects by month:")
month_breakdown = df_analysis['month'].value_counts().sort_index()
month_names = {1: 'Jan', 2: 'Feb', 3: 'Mar', 4: 'Apr', 5: 'May', 6: 'Jun',
               7: 'Jul', 8: 'Aug', 9: 'Sep', 10: 'Oct', 11: 'Nov', 12: 'Dec'}
for month in sorted(month_breakdown.index):
    print(f"  {month_names[month]} ({month:2d}): {month_breakdown[month]:,} projects")

df = df_analysis

print(f"\n✅ Date controls created successfully!")
print(f"   - Month fixed effects: C(month) - controls for seasonal patterns")
print(f"   - Year fixed effects: C(year) - controls for yearly trends")
print(f"   - This helps isolate the ChatGPT treatment effect from other time trends")


📅 Adding month and year controls...
   Date range: 2020-10-31 23:59:59 to 2024-12-10 23:59:59
   Years covered: [np.int32(2020), np.int32(2021), np.int32(2022), np.int32(2023), np.int32(2024)]
   Months covered: [np.int32(1), np.int32(2), np.int32(3), np.int32(4), np.int32(5), np.int32(6), np.int32(7), np.int32(8), np.int32(9), np.int32(10), np.int32(11), np.int32(12)]
   Unique month-year periods: 51

📊 Projects by year:
year
2020      969
2021     6807
2022    18650
2023    21215
2024    16847
Name: count, dtype: int64

📊 Projects by month:
  Jan ( 1): 5,775 projects
  Feb ( 2): 5,591 projects
  Mar ( 3): 6,221 projects
  Apr ( 4): 5,506 projects
  May ( 5): 5,557 projects
  Jun ( 6): 5,271 projects
  Jul ( 7): 5,300 projects
  Aug ( 8): 5,528 projects
  Sep ( 9): 5,217 projects
  Oct (10): 5,001 projects
  Nov (11): 5,305 projects
  Dec (12): 4,216 projects

✅ Date controls created successfully!
   - Month fixed effects: C(month) - controls for seasonal patterns
   - Year fixed effe

In [7]:
# Step 4: Create analysis sample

df_analysis = df[
    (df['currency'] == 'USD') &                  # keep one currency (avoid fx noise)
    (df['pledged'] > 0) &                        # drop projects that never raised a dime
    #(df['word_count'] > 500) &                   # remove small descriptions
    (df['text_quality'].notna()) &               # ensure quality metric available
    (df['created_at_parsed'].notna())            # valid launch date
].copy()

# Create treatment variables
gpt_date = pd.Timestamp('2022-11-30')
df_analysis['PostGPT'] = (df_analysis['created_at_parsed'] > gpt_date).astype(int)
df_analysis['indiegogo'] = (df_analysis['platform'] == 'Indiegogo').astype(int)
df_analysis['indiegogo_x_postgpt'] = df_analysis['indiegogo'] * df_analysis['PostGPT']

# Add controls
df_analysis['log_goal'] = np.log(df_analysis['goal'].clip(lower=1))

# control for year

print(f"✅ Analysis sample: {len(df_analysis):,} projects")
print("\nBreakdown:")
breakdown = pd.crosstab(df_analysis['platform'], df_analysis['PostGPT'], margins=True)
breakdown.columns = ['Pre-ChatGPT', 'Post-ChatGPT', 'Total']
print(breakdown)


✅ Analysis sample: 34,866 projects

Breakdown:
             Pre-ChatGPT  Post-ChatGPT  Total
platform                                     
Indiegogo           6617          4451  11068
Kickstarter         7405         16393  23798
All                14022         20844  34866


In [8]:
# Model 1: DID Regression - Text Quality Effects Only
# WHAT THIS TESTS: Did text quality importance change after ChatGPT? Did this differ by platform?

print("=" * 80)
print("MODEL 1: DIFFERENCE-IN-DIFFERENCES ANALYSIS (TEXT QUALITY ONLY)")
print("=" * 80)
print()
print("WHAT EACH VARIABLE TESTS:")
print("  text_quality = How much better text quality helps funding (baseline effect)")
print("  indiegogo = How much Indiegogo differs from Kickstarter (platform difference)")  
print("  PostGPT = How much funding changed after ChatGPT launch (time effect)")
print("  text_quality:indiegogo = Does text quality matter differently on Indiegogo vs Kickstarter?")
print("  text_quality:PostGPT = Did text quality become more/less important after ChatGPT? (KEY!)")
print("  indiegogo_x_postgpt = Did Indiegogo projects do better/worse after ChatGPT?")
print("  text_quality:indiegogo_x_postgpt = DID ESTIMATE: Platform difference in text quality change")
print()
print("CONTROLS:")
print("  log_goal = Project funding goal (bigger goals may attract more funding)")
print("  C(category_unified) = Project category (tech vs art vs games, etc.)")
print("  word_count = Description length (longer may signal more effort)")
print("  C(month_year) = Month-year fixed effects (controls for temporal trends)")
print()
print("INTERPRETATION GUIDE:")
print("  Positive coefficient = Variable increases funding")
print("  Negative coefficient = Variable decreases funding") 
print("  p < 0.05 = Statistically significant effect")
print("=" * 80)
print()

formula1 = """
log_pledged_amount ~ 
text_quality + indiegogo + PostGPT + 
text_quality:indiegogo + text_quality:PostGPT + 
indiegogo_x_postgpt + text_quality:indiegogo_x_postgpt +
log_goal + C(category_unified) + word_count + C(year):C(month)
"""

model1 = smf.ols(formula1.replace('\n', ' ').strip(), data=df_analysis).fit()
print(model1.summary())


MODEL 1: DIFFERENCE-IN-DIFFERENCES ANALYSIS (TEXT QUALITY ONLY)

WHAT EACH VARIABLE TESTS:
  text_quality = How much better text quality helps funding (baseline effect)
  indiegogo = How much Indiegogo differs from Kickstarter (platform difference)
  PostGPT = How much funding changed after ChatGPT launch (time effect)
  text_quality:indiegogo = Does text quality matter differently on Indiegogo vs Kickstarter?
  text_quality:PostGPT = Did text quality become more/less important after ChatGPT? (KEY!)
  indiegogo_x_postgpt = Did Indiegogo projects do better/worse after ChatGPT?
  text_quality:indiegogo_x_postgpt = DID ESTIMATE: Platform difference in text quality change

CONTROLS:
  log_goal = Project funding goal (bigger goals may attract more funding)
  C(category_unified) = Project category (tech vs art vs games, etc.)
  word_count = Description length (longer may signal more effort)
  C(month_year) = Month-year fixed effects (controls for temporal trends)

INTERPRETATION GUIDE:
  Pos

In [18]:
# Model 2: DID Regression - Adding AI Detection Scores  
# WHAT THIS TESTS: Same as Model 1, PLUS tests if AI-like writing explains the changes

print("\n" + "=" * 80)
print("MODEL 2: DIFFERENCE-IN-DIFFERENCES ANALYSIS (ADDING AI DETECTION)")
print("=" * 80)
print()
print("WHAT EACH VARIABLE TESTS:")
print("  text_quality = How much better text quality helps funding (baseline effect)")
print("  indiegogo = How much Indiegogo differs from Kickstarter (platform difference)")  
print("  PostGPT = How much funding changed after ChatGPT launch (time effect)")
print("  text_quality:indiegogo = Does text quality matter differently on Indiegogo vs Kickstarter?")
print("  text_quality:PostGPT = Did text quality become more/less important after ChatGPT? (KEY!)")
print("  indiegogo_x_postgpt = Did Indiegogo projects do better/worse after ChatGPT?")
print("  text_quality:indiegogo_x_postgpt = DID ESTIMATE: Platform difference in text quality change")
print()
print("NEW AI VARIABLES:")
print("  ai_ai_score = How much AI-like writing helps funding (baseline AI effect)")
print("  ai_ai_score:PostGPT = Did AI-like writing become more valuable after ChatGPT? (KEY!)")
print("  ai_ai_score:indiegogo_x_postgpt = DID for AI: Platform difference in AI writing value")
print()
print("CONTROLS:")
print("  log_goal = Project funding goal (bigger goals may attract more funding)")
print("  C(category_unified) = Project category (tech vs art vs games, etc.)")
print("  word_count = Description length (longer may signal more effort)")
print()
print("INTERPRETATION GUIDE:")
print("  Compare text_quality coefficients between Model 1 and Model 2:")
print("    - If they change → AI explains some of the text quality effects")
print("    - If AI coefficients are significant → AI has independent effects")
print("  Positive coefficient = Variable increases funding")
print("  Negative coefficient = Variable decreases funding")
print("  p < 0.05 = Statistically significant effect")
print("=" * 80)
print()

formula2 = """
log_pledged_amount ~ 
text_quality + indiegogo + PostGPT +
indiegogo:text_quality + 
indiegogo:PostGPT + indiegogo:PostGPT:text_quality +
text_quality:PostGPT + 
ai_score + ai_score:PostGPT +
log_goal + C(category_unified) + word_count + C(year):C(month)
"""
#ai_score:indiegogo_x_postgpt text_quality:indiegogo_x_postgpt +indiegogo_x_postgpt + text_quality:indiegogo +
model2 = smf.ols(formula2.replace('\n', ' ').strip(), data=df_analysis).fit()
print(model2.summary())



MODEL 2: DIFFERENCE-IN-DIFFERENCES ANALYSIS (ADDING AI DETECTION)

WHAT EACH VARIABLE TESTS:
  text_quality = How much better text quality helps funding (baseline effect)
  indiegogo = How much Indiegogo differs from Kickstarter (platform difference)
  PostGPT = How much funding changed after ChatGPT launch (time effect)
  text_quality:indiegogo = Does text quality matter differently on Indiegogo vs Kickstarter?
  text_quality:PostGPT = Did text quality become more/less important after ChatGPT? (KEY!)
  indiegogo_x_postgpt = Did Indiegogo projects do better/worse after ChatGPT?
  text_quality:indiegogo_x_postgpt = DID ESTIMATE: Platform difference in text quality change

NEW AI VARIABLES:
  ai_ai_score = How much AI-like writing helps funding (baseline AI effect)
  ai_ai_score:PostGPT = Did AI-like writing become more valuable after ChatGPT? (KEY!)
  ai_ai_score:indiegogo_x_postgpt = DID for AI: Platform difference in AI writing value

CONTROLS:
  log_goal = Project funding goal (bigg

In [20]:
# Compare Models - Key Coefficients Only
# WHAT THIS SHOWS: How adding AI scores changed the key results

print("\n" + "=" * 80)
print("MODEL COMPARISON: WHAT CHANGED WHEN WE ADDED AI SCORES?")
print("=" * 80)
print()
print("MODEL PERFORMANCE:")
print(f"  Model 1 R² (text only): {model1.rsquared:.4f}")  
print(f"  Model 2 R² (text + AI): {model2.rsquared:.4f}")
print(f"  Improvement: +{model2.rsquared - model1.rsquared:.4f}")
print()
print("KEY RESEARCH QUESTIONS - COEFFICIENT COMPARISON:")
print()

coeffs_to_show = [
    ('text_quality:PostGPT', 'Did text quality importance change after ChatGPT?'),
    ('text_quality:indiegogo_x_postgpt', 'DID ESTIMATE: Platform difference in text quality change'),
    ('ai_ai_score:PostGPT', 'Did AI-like writing become more valuable after ChatGPT?'),
    ('ai_ai_score:indiegogo_x_postgpt', 'DID for AI: Platform difference in AI writing value')
]

for coeff, description in coeffs_to_show:
    print(f"{description}")
    print(f"Variable: {coeff}")
    
    if coeff in model1.params:
        sig1 = "***" if model1.pvalues[coeff] < 0.001 else "**" if model1.pvalues[coeff] < 0.01 else "*" if model1.pvalues[coeff] < 0.05 else ""
        print(f"  Model 1: {model1.params[coeff]:+.4f} (p={model1.pvalues[coeff]:.3f}){sig1}")
    else:
        print(f"  Model 1: Not included")
    
    if coeff in model2.params:
        sig2 = "***" if model2.pvalues[coeff] < 0.001 else "**" if model2.pvalues[coeff] < 0.01 else "*" if model2.pvalues[coeff] < 0.05 else ""
        print(f"  Model 2: {model2.params[coeff]:+.4f} (p={model2.pvalues[coeff]:.3f}){sig2}")
        
        # Show change if both models have the coefficient
        if coeff in model1.params:
            change = model2.params[coeff] - model1.params[coeff]
            print(f"  Change:  {change:+.4f} {'(text effect changed when AI added)' if 'text_quality' in coeff else '(new AI effect)'}")
    else:
        print(f"  Model 2: Not included")
    print()

print("INTERPRETATION:")
print("  * = p < 0.05, ** = p < 0.01, *** = p < 0.001")
print("  If text_quality coefficients changed → AI explains some text quality effects")
print("  If AI coefficients are significant → AI has independent effects beyond text quality")
print("=" * 80)



MODEL COMPARISON: WHAT CHANGED WHEN WE ADDED AI SCORES?

MODEL PERFORMANCE:
  Model 1 R² (text only): 0.4538
  Model 2 R² (text + AI): 0.4564
  Improvement: +0.0025

KEY RESEARCH QUESTIONS - COEFFICIENT COMPARISON:

Did text quality importance change after ChatGPT?
Variable: text_quality:PostGPT
  Model 1: -5.8290 (p=0.000)***
  Model 2: -5.3399 (p=0.000)***
  Change:  +0.4891 (text effect changed when AI added)

DID ESTIMATE: Platform difference in text quality change
Variable: text_quality:indiegogo_x_postgpt
  Model 1: -69.8494 (p=0.427)
  Model 2: -68.4108 (p=0.435)
  Change:  +1.4386 (text effect changed when AI added)

Did AI-like writing become more valuable after ChatGPT?
Variable: ai_ai_score:PostGPT
  Model 1: Not included
  Model 2: Not included

DID for AI: Platform difference in AI writing value
Variable: ai_ai_score:indiegogo_x_postgpt
  Model 1: Not included
  Model 2: Not included

INTERPRETATION:
  * = p < 0.05, ** = p < 0.01, *** = p < 0.001
  If text_quality coeffic

In [ ]:
# Save results
df_analysis.to_csv('did_analysis_final_clean.csv', index=False)
df_analysis.to_pickle('did_analysis_final_clean.pkl')

print("✅ Results saved to:")
print("   - did_analysis_final_clean.csv")
print("   - did_analysis_final_clean.pkl")
